In [38]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langfuse import get_client
from pydantic import BaseModel
from typing import Literal

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGFUSE_PUBLIC_KEY'] = os.getenv('LANGFUSE_PUBLIC_KEY')
os.environ['LANGFUSE_SECRET_KEY'] = os.getenv('LANGFUSE_SECRET_KEY')
os.environ['LANGFUSE_HOST'] = os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
langfuse = get_client()

print("✓ Environment loaded")
print("✓ OpenAI client ready")
print("✓ Langfuse client ready")

✓ Environment loaded
✓ OpenAI client ready
✓ Langfuse client ready


In [39]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("financial-reconciliation")

with open("../data/processed/docstore.json", "r") as f:
    docstore = json.load(f)

print(f"✓ Pinecone connected")
print(f"✓ Docstore loaded: {len(docstore)} parents")
print(f"\nIndex stats:")
print(index.describe_index_stats())

✓ Pinecone connected
✓ Docstore loaded: 652 parents

Index stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL_PC': {'vector_count': 432},
                'MSFT_PC': {'vector_count': 989}},
 'total_vector_count': 1421,
 'vector_type': 'dense'}


In [40]:
bm25_encoder = BM25Encoder()
bm25_encoder.load("../data/processed/bm25_encoder.json")

print("✓ BM25 encoder loaded")

✓ BM25 encoder loaded


In [41]:
EMBEDDING_MODEL = "text-embedding-3-small"
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]


def get_embeddings(texts: list) -> list:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [e.embedding for e in response.data]


def scale_dense(vector: list, alpha: float) -> list:
    return [v * alpha for v in vector]


def scale_sparse(vector: dict, alpha: float) -> dict:
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }


class QueryRoute(BaseModel):
    chunk_type: Literal["table", "prose"]
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "MD&A", "Risk Factors", "Legal Proceedings",
        "Notes", "Shareholders Equity", "Any"
    ]


def route_query(query: str) -> dict:
    system_prompt = """You are a financial document query router for SEC 10-Q filings.

PRIORITY RULES — apply these first:
- Question contains "inventor" (inventory/inventories) → ALWAYS Balance Sheet, table
- Question asks about dividends PAID as cash outflow → ALWAYS Cash Flow, table
- Question asks for R&D as percentage or ratio → ALWAYS MD&A, prose
- Question asks for EPS computation or weighted average shares → ALWAYS Notes, table
- Questions with "change", "grew", "increased", "decreased" about a metric → MD&A, prose
  UNLESS asking for raw dollar figure with "what was" → Income Statement, table
- "gross margin" as dollar figure → Income Statement, table
- "R&D expense" or "research and development expense" as dollar amount → Income Statement, table
- "operating expenses" as dollar figure → Income Statement, table

SECTION GUIDE:
- "Income Statement": revenue, net sales, gross margin, operating income, net income, EPS raw figure, cost of sales, R&D dollar amount, operating expenses
- "Balance Sheet": total assets, cash, liabilities, equity total, receivables, inventory
- "Cash Flow": operating activities, investing, financing, cash generated, dividends paid, capex
- "MD&A": why metrics changed, segment performance, growth reasons, percentage change, ratios
- "Risk Factors": risks, uncertainties, challenges
- "Legal Proceedings": lawsuits, investigations, legal matters
- "Notes": share repurchase details, RSUs, debt details, EPS computation, weighted average shares
- "Shareholders Equity": retained earnings change, AOCI, equity statement details
- "Any": spans multiple sections

CHUNK TYPE RULES:
- Specific NUMBER or FIGURE → table
- EXPLANATION, REASON, WHY, HOW → prose
- Percentage change, growth rate → MD&A, prose

Return only valid JSON matching the schema."""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        response_format=QueryRoute,
        temperature=0
    )
    route = response.choices[0].message.parsed
    return {"chunk_type": route.chunk_type, "section": route.section}


print("✓ get_embeddings()")
print("✓ scale_dense() + scale_sparse()")
print("✓ route_query() with structured output")

✓ get_embeddings()
✓ scale_dense() + scale_sparse()
✓ route_query() with structured output


In [42]:
def retrieve_with_parent_child(
    query: str,
    ticker: str,
    section: str = None,
    chunk_type: str = None,
    top_k: int = 5,
    alpha: float = 0.5
) -> list:

    namespace = f"{ticker}_PC"
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder.encode_queries(query)

    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []

    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break

    return retrieved


def retrieve_with_router(
    query: str,
    ticker: str,
    top_k: int = 5,
    alpha: float = 0.5
) -> tuple:

    route = route_query(query)
    section = route["section"]
    chunk_type = route["chunk_type"]

    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=section,
        chunk_type=chunk_type,
        top_k=top_k,
        alpha=alpha
    )
    return retrieved, route


# Quick test
query = "What was Apple total net sales in Q2 2026?"
results, route = retrieve_with_router(query, "AAPL", top_k=3)

print(f"Query : {query}")
print(f"Route : {route}")
print(f"Results: {len(results)}")
for i, r in enumerate(results, 1):
    print(f"\nResult {i} | score: {r['child_score']:.4f}")
    print(f"  Section: {r['metadata'].get('section')}")
    print(f"  Child  : {r['child_content'][:100]}")

Query : What was Apple total net sales in Q2 2026?
Route : {'chunk_type': 'table', 'section': 'Income Statement'}
Results: 2

Result 1 | score: 0.3210
  Section: Income Statement
  Child  : |                                              | Three Months Ended   | Three Months Ended   | Six M

Result 2 | score: 0.2334
  Section: Income Statement
  Child  : |                                                                              | Three Months Ended 


In [43]:
def answer_question(query: str, ticker: str, top_k: int = 3, alpha: float = 0.5) -> dict:
    """
    Full pipeline: route → retrieve → prompt → generate → return
    No tracing — baseline function
    """
    start_time = time.time()

    # Step 1 — Route
    route = route_query(query)

    # Step 2 — Retrieve
    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=route["section"],
        chunk_type=route["chunk_type"],
        top_k=top_k,
        alpha=alpha
    )

    # Step 3 — Build context
    context = "\n\n---\n\n".join([
        f"[Source: {r['metadata'].get('section')} | "
        f"Page {r['metadata'].get('page')} | "
        f"{r['metadata'].get('chunk_type')}]\n"
        f"{r['parent_content']}"
        for r in retrieved
    ])

    # Step 4 — System prompt
    system_prompt = """You are a financial analyst assistant specializing in SEC filings.

Answer questions based ONLY on the provided financial document context.
Rules:
- State numbers exactly as they appear in the document
- Financial figures in 10-Q tables are in millions unless stated otherwise
- Always mention the exact period: "Three Months Ended March 28, 2026"
- 10-Q tables show BOTH quarterly (Three Months Ended) AND year-to-date (Six Months Ended)
- When asked about a specific quarter → use Three Months Ended column ONLY
- Always append "million" to dollar figures from financial statements
- If you cannot find the answer say: "Not found in the provided filing sections"
- Never fabricate numbers"""

    user_prompt = f"""Financial filing context:
{context}

Question: {query}

Answer:"""

    # Step 5 — Generate
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content
    total_time = time.time() - start_time

    return {
        "query": query,
        "ticker": ticker,
        "answer": answer,
        "route": route,
        "sources": [
            {
                "section": r["metadata"].get("section"),
                "page": r["metadata"].get("page"),
                "chunk_type": r["metadata"].get("chunk_type"),
                "child_score": r["child_score"]
            }
            for r in retrieved
        ],
        "usage": {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        },
        "latency_ms": round(total_time * 1000)
    }


# Test
result = answer_question("What was Apple total net sales in Q2 2026?", "AAPL")

print(f"Query  : {result['query']}")
print(f"Answer : {result['answer']}")
print(f"Latency: {result['latency_ms']}ms")
print(f"Tokens : {result['usage']}")
print(f"\nSources:")
for s in result['sources']:
    print(f"  - {s['section']} | page {s['page']} | score {s['child_score']:.4f}")

Query  : What was Apple total net sales in Q2 2026?
Answer : Apple's total net sales in Q2 2026 (Three Months Ended March 28, 2026) was $ 111,184 million.
Latency: 10449ms
Tokens : {'input_tokens': 1434, 'output_tokens': 32, 'total_tokens': 1466}

Sources:
  - Income Statement | page 4.0 | score 0.3210
  - Income Statement | page 5.0 | score 0.2334


In [44]:
def answer_question_traced(query: str, ticker: str, top_k: int = 3, alpha: float = 0.5) -> dict:
    """Full pipeline with Langfuse observability."""
    start_time = time.time()

    with langfuse.start_as_current_observation(
        as_type="span",
        name="answer_question",
        input={"query": query, "ticker": ticker}
    ) as root_span:

        # Step 1 — Route
        with langfuse.start_as_current_observation(
            as_type="span",
            name="route_query",
            input={"query": query}
        ) as route_span:
            route = route_query(query)
            route_span.update(output=route)

        # Step 2 — Retrieve
        with langfuse.start_as_current_observation(
            as_type="span",
            name="retrieve_chunks",
            input={
                "query": query,
                "ticker": ticker,
                "section": route["section"],
                "chunk_type": route["chunk_type"]
            }
        ) as retrieve_span:
            retrieved = retrieve_with_parent_child(
                query=query,
                ticker=ticker,
                section=route["section"],
                chunk_type=route["chunk_type"],
                top_k=top_k,
                alpha=alpha
            )
            retrieve_span.update(output={
                "chunks_retrieved": len(retrieved),
                "sections": [r["metadata"].get("section") for r in retrieved]
            })

        # Step 3 — Build context
        context = "\n\n---\n\n".join([
            f"[Source: {r['metadata'].get('section')} | "
            f"Page {r['metadata'].get('page')} | "
            f"{r['metadata'].get('chunk_type')}]\n"
            f"{r['parent_content']}"
            for r in retrieved
        ])

        # Step 4 — Fetch prompt from Langfuse
        try:
            prompt_obj = langfuse.get_prompt(
                "financial_analyst_system",
                label="production"
            )
            system_prompt = prompt_obj.prompt
            prompt_version = prompt_obj.version
        except:
            # Fallback if prompt not in Langfuse yet
            system_prompt = """You are a financial analyst assistant specializing in SEC filings.
Answer questions based ONLY on the provided financial document context.
Rules:
- State numbers exactly as they appear in the document
- Financial figures in 10-Q tables are in millions unless stated otherwise
- Always mention the exact period: "Three Months Ended March 28, 2026"
- 10-Q tables show BOTH quarterly (Three Months Ended) AND year-to-date (Six Months Ended)
- When asked about a specific quarter → use Three Months Ended column ONLY
- Always append "million" to dollar figures from financial statements
- If you cannot find the answer say: "Not found in the provided filing sections"
- Never fabricate numbers"""
            prompt_version = 0

        user_prompt = f"""Financial filing context:
{context}

Question: {query}

Answer:"""

        # Step 5 — Generate
        with langfuse.start_as_current_observation(
            as_type="generation",
            name="llm_generation",
            model="gpt-4o-mini",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        ) as gen_span:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )
            answer = response.choices[0].message.content
            gen_span.update(
                output=answer,
                usage_details={
                    "input": response.usage.prompt_tokens,
                    "output": response.usage.completion_tokens
                }
            )

        total_time = time.time() - start_time
        root_span.update(output={"answer": answer})

    langfuse.flush()

    return {
        "query": query,
        "ticker": ticker,
        "answer": answer,
        "route": route,
        "prompt_version": prompt_version,
        "sources": [
            {
                "section": r["metadata"].get("section"),
                "page": r["metadata"].get("page"),
                "chunk_type": r["metadata"].get("chunk_type"),
                "child_score": r["child_score"]
            }
            for r in retrieved
        ],
        "usage": {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        },
        "latency_ms": round(total_time * 1000)
    }


# Test
result = answer_question_traced(
    "What was Apple total net sales in Q2 2026?",
    "AAPL"
)

print(f"Query         : {result['query']}")
print(f"Answer        : {result['answer']}")
print(f"Prompt version: {result['prompt_version']}")
print(f"Latency       : {result['latency_ms']}ms")
print(f"Tokens        : {result['usage']}")
print(f"\nCheck Langfuse dashboard → Tracing")

Query         : What was Apple total net sales in Q2 2026?
Answer        : Apple's total net sales for the "Three Months Ended March 28, 2026" was $111,184 million.
Prompt version: 4
Latency       : 5815ms
Tokens        : {'input_tokens': 1525, 'output_tokens': 27, 'total_tokens': 1552}

Check Langfuse dashboard → Tracing


In [45]:
test_queries = [
    ("What was Apple total net sales in Q2 2026?", "AAPL"),
    ("What was Apple net income in Q2 2026?", "AAPL"),
    ("What was Apple gross margin in Q2 2026?", "AAPL"),
    ("Why did Apple iPhone revenue increase in Q2 2026?", "AAPL"),
    ("What was Apple cash and cash equivalents?", "AAPL"),
    ("What was Apple operating income in Q2 2026?", "AAPL"),
    ("What was Microsoft net income in Q1 FY2026?", "MSFT"),
    ("What was Microsoft total revenue in Q1 FY2026?", "MSFT"),
]

print("Running test queries...\n")
print(f"{'Query':<55} {'Answer':<50} {'ms':>6} {'tokens':>7}")
print("-" * 125)

for query, ticker in test_queries:
    result = answer_question_traced(query, ticker)
    print(f"{query[:54]:<55} {result['answer'][:49]:<50} {result['latency_ms']:>6} {result['usage']['total_tokens']:>7}")

print("\n✓ All queries traced — check Langfuse dashboard")

Running test queries...

Query                                                   Answer                                                 ms  tokens
-----------------------------------------------------------------------------------------------------------------------------
What was Apple total net sales in Q2 2026?              Total net sales for Apple in the "Three Months En    3642    1552
What was Apple net income in Q2 2026?                   Apple's net income for the Three Months Ended Mar    2624    1550
What was Apple gross margin in Q2 2026?                 The gross margin for Apple in the "Three Months E    2949    1551
Why did Apple iPhone revenue increase in Q2 2026?       Not found in the provided filing sections.           2564     905
What was Apple cash and cash equivalents?               For the period "Three Months Ended March 28, 2026    2696    1065
What was Apple operating income in Q2 2026?             Apple's operating income for the period "Three Mo    2679    

In [ ]:
# Cell 9 — FastAPI app
# We write this to src/api/main.py
# Then run it from terminal

fastapi_code = '''
import os
import json
import time
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langfuse import get_client

load_dotenv()

# ── Globals ──────────────────────────────────────────────
openai_client = None
index = None
docstore = None
bm25_encoder = None
langfuse = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load all resources on startup."""
    global openai_client, index, docstore, bm25_encoder, langfuse

    print("Loading resources...")

    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    index = pc.Index("financial-reconciliation")

    with open("../../data/processed/docstore.json", "r") as f:
        docstore = json.load(f)

    bm25_encoder = BM25Encoder()
    bm25_encoder.load("../../data/processed/bm25_encoder.json")

    langfuse = get_client()

    print(f"✓ Resources loaded | Docstore: {len(docstore)} parents")
    yield
    print("Shutting down...")


app = FastAPI(
    title="Financial Reconciliation Agent",
    description="RAG system for SEC 10-Q financial filings",
    version="1.0.0",
    lifespan=lifespan
)

# ── Request/Response schemas ──────────────────────────────
class QueryRequest(BaseModel):
    query: str
    ticker: str
    top_k: Optional[int] = 3
    alpha: Optional[float] = 0.5

class Source(BaseModel):
    section: str
    page: float
    chunk_type: str
    child_score: float

class QueryResponse(BaseModel):
    query: str
    ticker: str
    answer: str
    route: dict
    sources: list[Source]
    usage: dict
    latency_ms: int
    prompt_version: int

# ── Helper functions ──────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]

def get_embeddings(texts):
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=texts
    )
    return [e.embedding for e in response.data]

def scale_dense(vector, alpha):
    return [v * alpha for v in vector]

def scale_sparse(vector, alpha):
    return {"indices": vector["indices"],
            "values": [v * (1 - alpha) for v in vector["values"]]}

class QueryRoute(BaseModel):
    chunk_type: str
    section: str

def route_query(query: str) -> dict:
    from pydantic import BaseModel
    from typing import Literal

    class QRoute(BaseModel):
        chunk_type: Literal["table", "prose"]
        section: Literal[
            "Income Statement", "Balance Sheet", "Cash Flow",
            "MD&A", "Risk Factors", "Legal Proceedings",
            "Notes", "Shareholders Equity", "Any"
        ]

    system_prompt = """You are a financial document query router for SEC 10-Q filings.
PRIORITY RULES:
- inventory/inventories → Balance Sheet, table
- dividends PAID → Cash Flow, table
- R&D as percentage → MD&A, prose
- EPS computation → Notes, table
- gross margin as dollar → Income Statement, table
- R&D dollar amount → Income Statement, table

SECTION GUIDE:
- Income Statement: revenue, net sales, gross margin, operating income, net income, cost of sales
- Balance Sheet: total assets, cash, liabilities, equity, receivables, inventory
- Cash Flow: operating activities, investing, financing, dividends paid, capex
- MD&A: why metrics changed, percentage change, growth reasons
- Risk Factors: risks, uncertainties
- Legal Proceedings: lawsuits, investigations
- Notes: RSUs, debt details, EPS computation, weighted average shares
- Shareholders Equity: retained earnings, AOCI
- Any: spans multiple sections

CHUNK TYPE: NUMBER/FIGURE → table | EXPLANATION/WHY/HOW → prose"""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        response_format=QRoute,
        temperature=0
    )
    route = response.choices[0].message.parsed
    return {"chunk_type": route.chunk_type, "section": route.section}

def retrieve(query, ticker, section, chunk_type, top_k, alpha):
    namespace = f"{ticker}_PC"
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder.encode_queries(query)

    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []
    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break
    return retrieved

# ── Routes ────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "ok",
        "docstore_parents": len(docstore),
        "index": "financial-reconciliation"
    }

@app.post("/query", response_model=QueryResponse)
def query_endpoint(request: QueryRequest):
    start_time = time.time()

    with langfuse.start_as_current_observation(
        as_type="span",
        name="answer_question",
        input={"query": request.query, "ticker": request.ticker}
    ) as root_span:

        # Route
        with langfuse.start_as_current_observation(
            as_type="span", name="route_query",
            input={"query": request.query}
        ) as route_span:
            route = route_query(request.query)
            route_span.update(output=route)

        # Retrieve
        with langfuse.start_as_current_observation(
            as_type="span", name="retrieve_chunks",
            input={"ticker": request.ticker, **route}
        ) as retrieve_span:
            retrieved = retrieve(
                request.query, request.ticker,
                route["section"], route["chunk_type"],
                request.top_k, request.alpha
            )
            retrieve_span.update(output={"chunks_retrieved": len(retrieved)})

        # Build context
        context = "\\n\\n---\\n\\n".join([
            f"[{r[\'metadata\'].get(\'section\')} | Page {r[\'metadata\'].get(\'page\')}]\\n{r[\'parent_content\']}"
            for r in retrieved
        ])

        # Fetch prompt
        try:
            prompt_obj = langfuse.get_prompt("financial_analyst_system", label="production")
            system_prompt = prompt_obj.prompt
            prompt_version = prompt_obj.version
        except:
            system_prompt = """You are a financial analyst assistant.
Answer based ONLY on context provided.
Use Three Months Ended column for quarterly questions.
Append million to dollar figures. Never fabricate numbers."""
            prompt_version = 0

        user_prompt = f"Context:\\n{context}\\n\\nQuestion: {request.query}\\n\\nAnswer:"

        # Generate
        with langfuse.start_as_current_observation(
            as_type="generation", name="llm_generation",
            model="gpt-4o-mini",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        ) as gen_span:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )
            answer = response.choices[0].message.content
            gen_span.update(
                output=answer,
                usage_details={
                    "input": response.usage.prompt_tokens,
                    "output": response.usage.completion_tokens
                }
            )

        root_span.update(output={"answer": answer})

    langfuse.flush()
    total_time = time.time() - start_time

    return QueryResponse(
        query=request.query,
        ticker=request.ticker,
        answer=answer,
        route=route,
        sources=[
            Source(
                section=r["metadata"].get("section", ""),
                page=r["metadata"].get("page", 0),
                chunk_type=r["metadata"].get("chunk_type", ""),
                child_score=r["child_score"]
            )
            for r in retrieved
        ],
        usage={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        },
        latency_ms=round(total_time * 1000),
        prompt_version=prompt_version
    )
'''




UnicodeEncodeError: 'charmap' codec can't encode characters in position 427-428: character maps to <undefined>

: 